# 04 — Damir: OCR, Business Parameters & Terms

**Google Colab notebook.** Runtime → *Change runtime type* → **GPU (T4 / L4 / A100)** before running.

Runs GPU OCR over invoices/receipts, scores it against **real transcriptions**, then checks business-parameter presence and extracts payment terms.

| | |
|---|---|
| **Inputs** | Drive: `datasets/ocr_multitype/`, `inputs/annotations/`, `inputs/images/` |
| **Outputs** | `ocr_outputs.csv`, `parameter_presence_results.csv`, `terms_extraction_results.csv`, metrics JSON |
| **Expected runtime** | ~40–70 min on a T4 |
| **Compute profile** | `colab_gpu` (generous — full data, pinned in the profile cell) |

### How results get back to the team
Everything is written to Google Drive by `colab_bootstrap.publish()`, into **both**:
- `outputs/damir/<kind>/` — the *latest* copy
- `runs/damir/<UTC-timestamp>/<kind>/` — an immutable archive, so re-running never
  silently destroys an earlier result

Tell the integrator (Hessam) when you're done; he copies from `outputs/` into the repo.

> **Before you run:** `MyDrive/DL2_InvoiceAI/` must already contain `code/` (the repo's `src/`,
> `scripts/`, and `colab_bootstrap.py`) and `inputs/`. If it doesn't, the bootstrap cell fails
> fast with a message telling you exactly what's missing.

### You have two evaluation sets, with very different strength

| | OCR Dataset (**primary**) | Batch CSVs (**secondary**) |
|---|---|---|
| Images | 973, pre-split | 5,201, of which **1,413 annotated** |
| Coverage | **100%** | ~27% |
| Text GT | per-box transcription (52,331) | page-level blob |
| Fields | company, date, address, total | invoice, items, subtotal, payment_instructions |

Report the **primary** numbers as your headline — they have full coverage and per-box text, so
CER/WER is meaningful. Use the batch subset as a secondary, real-full-page-invoice check, and
**always state the denominator** (only ~197 of the 750 manifest images have any GT at all).

### Don't reimplement the shared logic
`src/parameter_checker.py` and `src/terms_extraction.py` already exist and are unit-tested. Import
them. If you find a real bug, report it rather than forking the logic.

In [ ]:
# --- GPU check: stop here if this says "no GPU" ---------------------------------
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "!! NO GPU. Runtime > Change runtime type > Hardware accelerator = GPU, then re-run.")
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# --- Mount Drive + load the shared bootstrap -----------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs

import sys, os, shutil, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

_bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
assert _bs.exists(), (
    f"Missing {_bs}.\nUpload the repo's colab/colab_bootstrap.py into "
    f"{DRIVE_ROOT}/code/ and re-run this cell."
)
sys.path.insert(0, str(_bs.parent))
import colab_bootstrap as CB

root  = CB.mount_drive(DRIVE_ROOT)
paths = CB.setup_paths(root)
CB.install_deps("easyocr", "rapidfuzz", "pandas", "opencv-python-headless", "jiwer")
print("Drive root:", root)

In [ ]:
# --- Pin the generous Colab budget --------------------------------------------
os.environ["IIP_COMPUTE_PROFILE"] = "colab_gpu"
from src.compute_profile import get_profile

P = get_profile()
print(json.dumps(P, indent=2, default=str))

RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())   # one archive folder for this run
T0 = time.time()

In [ ]:
# --- Datasets: read straight from Google Drive (NO Kaggle token needed) ------
# Primary eval set. The secondary invoice check reads inputs/images/ + inputs/annotations/, which are already in Drive.
# Paths are resolved tolerantly: if a dataset was copied one level too deep
# (e.g. ocr_multitype/invoice/train/... instead of ocr_multitype/train/...),
# it is found anyway and a NOTE is printed. No re-upload needed.
DATA = paths.inputs / "datasets"

BASE = CB.resolve_dataset_root(DATA / "ocr_multitype", ['train/annotations', 'val/annotations', 'test/annotations'])

print(f"  ocr_multitype  -> {BASE}")
print(f"                    " f"{sum(1 for _ in BASE.rglob(chr(42)) if _.is_file()):,} files")

In [ ]:
# --- Load the primary GT (per-box transcriptions + entities) -----------------
import pandas as pd, numpy as np, cv2

def load(sp):
    out = []
    for ap in sorted((BASE/sp/"annotations").glob("*.json")):
        d = json.loads(ap.read_text(encoding="utf-8"))
        img = next((BASE/sp/"images").glob(ap.stem+".*"), None)
        if img:
            out.append({"file_id": d.get("file_id", ap.stem), "img": img,
                        "entities": d.get("entities", {}),
                        "text": "\n".join(b.get("text", "") for b in d.get("ocr_boxes", []))})
    return out
test = load("test")
print("primary eval docs:", len(test), "(100% have GT text + entities)")
print("sample GT text:\n", test[0]["text"][:300])

In [ ]:
# --- GPU OCR ------------------------------------------------------------------
import easyocr, time
reader = easyocr.Reader(["en"], gpu=True)

N = P.get("max_images_per_class") or len(test)
rows, t0 = [], time.time()
for r in test[:N]:
    res = reader.readtext(str(r["img"]), detail=1, paragraph=False)
    txt = "\n".join(t for _, t, _ in res)
    conf = float(np.mean([c for _, _, c in res])) if res else 0.0
    rows.append({"document_id": r["file_id"], "image_path": str(r["img"]),
                 "ocr_text": txt, "mean_confidence": round(conf, 4),
                 "n_boxes": len(res), "source": "ocr_dataset_test"})
ocr = pd.DataFrame(rows)
print(f"OCR'd {len(ocr)} docs in {time.time()-t0:.0f}s "
      f"({(time.time()-t0)/max(len(ocr),1):.2f}s/doc)")
ocr.head(2)

In [ ]:
# --- Score OCR against the real transcriptions (CER / WER) -------------------
import jiwer
gtmap = {r["file_id"]: r["text"] for r in test}

def clean(s):
    return " ".join(str(s).upper().split())

cers, wers = [], []
for r in ocr.itertuples():
    g, h = clean(gtmap.get(r.document_id, "")), clean(r.ocr_text)
    if not g:
        continue
    cers.append(jiwer.cer(g, h))
    wers.append(jiwer.wer(g, h))

ocr_metrics = {
    "n_scored": len(cers),
    "cer_mean": round(float(np.mean(cers)), 4) if cers else None,
    "cer_median": round(float(np.median(cers)), 4) if cers else None,
    "wer_mean": round(float(np.mean(wers)), 4) if wers else None,
    "wer_median": round(float(np.median(wers)), 4) if wers else None,
}
print(json.dumps(ocr_metrics, indent=2))
print("\nLower is better. CER ~0.1 = roughly 1 character in 10 wrong.")

In [ ]:
# --- Run OCR on ALL 750 invoices, keyed by document_id (feeds readiness) -------
# CORE FIX (step C) + a Drive CACHE so re-running step-E/F tweaks does NOT re-OCR
# (OCR takes ~15-40 min). Delete the cache file printed below to force a fresh OCR.
man = pd.read_csv(paths.inputs / "invoice_manifest.csv")

# batch1 GT -> a secondary CER number (circular-ish; always report its denominator)
ann = sorted((paths.inputs / "annotations").glob("batch1_*.csv"))
gt2 = None
if ann:
    gt2 = pd.concat([pd.read_csv(p) for p in ann], ignore_index=True)
    gt2["stem"] = gt2["File Name"].astype(str).str.replace(".jpg", "", regex=False).str.strip()
    gt2 = gt2.drop_duplicates("stem").set_index("stem")

CACHE = paths.outputs("damir") / "invoice_ocr_cache.csv"
if CACHE.exists():
    inv = pd.read_csv(CACHE)
    inv["ocr_text"] = inv["ocr_text"].fillna("")
    print(f"loaded cached invoice OCR: {len(inv)} rows  (delete {CACHE} to force re-OCR)")
else:
    inv_rows, t0 = [], time.time()
    for r in man.itertuples():
        ip = root / r.image_path
        if not ip.exists():
            continue
        res = reader.readtext(str(ip), detail=1, paragraph=False)
        txt = "\n".join(t for _, t, _ in res)
        conf = float(np.mean([c for _, _, c in res])) if res else 0.0
        inv_rows.append({"document_id": r.document_id, "image_path": r.image_path,
                         "ocr_text": txt, "mean_confidence": round(conf, 4),
                         "n_boxes": len(res), "source": "invoice"})
    inv = pd.DataFrame(inv_rows)
    inv.to_csv(CACHE, index=False)
    print(f"OCR'd + cached {len(inv)} / {len(man)} invoices in {time.time()-t0:.0f}s")

cers2 = []
if gt2 is not None:
    for r in inv.itertuples():
        if r.document_id in gt2.index:
            ref = clean(str(gt2.loc[r.document_id, "OCRed Text"]))
            if ref:
                cers2.append(jiwer.cer(ref, clean(str(r.ocr_text))))
sec_metrics = {"n_scored": len(cers2),
               "cer_mean": round(float(np.mean(cers2)), 4) if cers2 else None,
               "denominator_note": f"{len(cers2)} of {len(man)} invoices had batch GT text"}
print("secondary invoice CER:", json.dumps(sec_metrics))


In [ ]:
# --- Business-parameter presence: REAL module + TIGHTENED matching (step E) ----
# The v2 config had catch-all patterns and a 2-letter "po" keyword, so 3 fields hit a
# false 100% and PO was inflated by words like "report"/"deposit". These rules require
# real context + digits, so presence reflects GENUINE references (precision over recall,
# right for a compliance gate). The config is OVERWRITTEN so the new patterns take effect.
REQUIRED_FIELDS_CONFIG = {
    "default_required_fields": [
        {"field_name": "PO Reference", "required": True,
         "keywords": ["purchase order", "p.o.", "po no", "po number", "po ref"],
         "patterns": [r"\bPO[-\s#:]?\d{2,}", r"\bP\.O\.[-\s#:]*\d{2,}"]},
        {"field_name": "Order Number", "required": True,
         "keywords": ["order number", "order no.", "sales order"],
         "patterns": [r"\bORD[-\s#:]?\d{2,}", r"\border\s*(?:no\.?|number|#|:)\s*[-#:]?\s*\d{2,}"]},
        {"field_name": "Contract Number", "required": False,
         "keywords": ["contract no", "contract number", "agreement no"],
         "patterns": [r"\bCN[-\s#:]?\d{2,}", r"\bcontract\s*(?:no\.?|#|:)\s*\d{2,}"]},
        {"field_name": "Work Order No.", "required": False,
         "keywords": ["work order", "job order"],
         "patterns": [r"\bWO[-\s#:]?\d{2,}", r"\bJOB[-\s#:]?\d{2,}"]},
        {"field_name": "Project Reference", "required": False,
         "keywords": ["project ref", "project number", "project code"],
         "patterns": [r"\bPRJ[-\s#:]?\d{2,}", r"\bproject\s*(?:no\.?|#|code|:)\s*\d{2,}"]},
        {"field_name": "Insurance Policy Number", "required": False,
         "keywords": ["insurance policy", "policy no", "policy number"],
         "patterns": [r"\bpolicy\s*(?:no\.?|number|#|:)\s*[-#:]?\s*[A-Z0-9][A-Z0-9-]{3,}"]},
        {"field_name": "Bill of Lading Number", "required": False,
         "keywords": ["bill of lading", "b/l no", "bl no"],
         "patterns": [r"\bB/?L[-\s#:]?(?:no\.?|#|:)?\s*\d{3,}"]},
    ],
    "custom_fields": [],
}
cfg_dir = Path(root) / "code" / "config"; cfg_dir.mkdir(parents=True, exist_ok=True)
cfg_path = cfg_dir / "required_fields_config.json"
cfg_path.write_text(json.dumps(REQUIRED_FIELDS_CONFIG, indent=2), encoding="utf-8")   # OVERWRITE
print("wrote tightened config ->", cfg_path)

# load_required_fields() reads this file fresh on each call, so the new rules apply now.
from src import parameter_checker as PC
prows = []
for r in inv.itertuples():
    for res in PC.check_all_fields(r.ocr_text or ""):
        prows.append({"document_id": r.document_id, **res})
pres = pd.DataFrame(prows, columns=["document_id", "field_name", "required",
                                    "present", "matched_text", "match_method"])
pres.to_csv("/content/out_parameter_presence_results.csv", index=False)

req_any = pres[pres.required].groupby("document_id").present.any()
print("invoices with >=1 REQUIRED reference present:",
      f"{req_any.mean():.1%} ({int(req_any.sum())}/{len(req_any)})")
print("per-field present rate (%):")
print(pres.groupby("field_name").present.mean().mul(100).round(1).sort_values(ascending=False).to_string())

# --- AUDIT: eyeball real matches (anti-overfitting check) ---------------------
print("\n--- matched_text audit (do these look like real references?) ---")
aud = (pres[pres.present & pres.matched_text.notna()]
       .groupby("field_name").head(5)[["field_name", "matched_text", "match_method"]])
print(aud.to_string(index=False) if len(aud) else "  (no positive matches)")


In [ ]:
# --- Terms & payment extraction: REAL module, per invoice (contract schema) ----
# FIX (step A): call terms_extraction.extract_terms_and_conditions -- the notebook was
# calling non-existent names and always got {} (0% terms). Whole-page OCR text is fed to
# the region slots so dates / terms / clauses are searched across the full invoice.
from src import terms_extraction as TE

def _terms_row(doc_id, text):
    text = text or ""
    rt = {"date_region": text, "due_date_region": "",
          "payment_terms_region": text, "terms_and_conditions_region": text}
    o = TE.extract_terms_and_conditions(rt)
    pcx, tc = o["payment_context"], o["terms_and_conditions"]
    return {"document_id": doc_id,
            "invoice_date": pcx["invoice_date"], "due_date": pcx["due_date"],
            "payment_terms": pcx["payment_terms"], "billing_due_days": pcx["billing_due_days"],
            "late_payment_flag": tc["late_payment_clause_detected"],
            "dispute_flag": tc["dispute_clause_detected"],
            "penalty_flag": tc["penalty_clause_detected"],
            "extracted_text": tc["extracted_text"], "summary": tc["summary"]}

tdf = pd.DataFrame([_terms_row(r.document_id, r.ocr_text) for r in inv.itertuples()])
tdf.to_csv("/content/out_terms_extraction_results.csv", index=False)
flags = tdf[["late_payment_flag", "dispute_flag", "penalty_flag"]].any(axis=1)
print("date found:", f"{tdf.invoice_date.notna().mean():.1%}",
      "| payment_terms found:", f"{tdf.payment_terms.notna().mean():.1%}",
      "| billing_due_days:", f"{tdf.billing_due_days.notna().mean():.1%}",
      "| any clause:", f"{flags.mean():.1%}")


In [ ]:
# --- Provenance: the _run block makes cross-run model comparison possible ------
def run_block(**kw):
    """Stamp every metrics JSON with how it was produced, so local-CPU and Colab-GPU
    results can be charted against each other later."""
    b = {
        "profile": P.get("profile_name", "colab_gpu"),
        "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
        "epochs": P.get("epochs"), "imgsz": P.get("imgsz"), "batch": P.get("batch"),
        "wall_clock_sec": round(time.time() - T0, 1),
        "timestamp_utc": RUN_TS, "member": "damir",
    }
    b.update(kw)
    return b


In [ ]:
# --- Write the four contracted outputs (now INVOICE-keyed) --------------------
OUTD = Path("/content/out"); OUTD.mkdir(exist_ok=True)

# ocr_outputs.csv: receipts (for CER/WER eval) + all 750 invoices
ocr_all = pd.concat([ocr, inv], ignore_index=True)
ocr_all.to_csv(OUTD / "ocr_outputs.csv", index=False)
shutil.copyfile("/content/out_parameter_presence_results.csv", OUTD / "parameter_presence_results.csv")
shutil.copyfile("/content/out_terms_extraction_results.csv", OUTD / "terms_extraction_results.csv")

req_any = pres[pres.required].groupby("document_id").present.any()
param_summary = {
    "per_field_present_rate": pres.groupby("field_name").present.mean().round(4).to_dict(),
    "invoices_with_required_reference": round(float(req_any.mean()), 4),
}
terms_summary = {
    "invoice_date_rate": round(float(tdf.invoice_date.notna().mean()), 4),
    "payment_terms_rate": round(float(tdf.payment_terms.notna().mean()), 4),
    "billing_due_days_rate": round(float(tdf.billing_due_days.notna().mean()), 4),
    "any_clause_rate": round(float(tdf[["late_payment_flag", "dispute_flag", "penalty_flag"]].any(axis=1).mean()), 4),
}
met = Path("/content/out/ocr_parameter_metrics.json")
met.write_text(json.dumps({
    "ocr_primary_receipts": ocr_metrics,
    "ocr_secondary_invoices": sec_metrics,
    "invoice_parameter_presence": param_summary,
    "invoice_terms_extraction": terms_summary,
    "n_invoices": int(len(inv)),
    "_run": run_block(model="easyocr-en", engine="easyocr",
                      eval_set="receipts (CER/WER) + 750 invoices (params/terms)"),
}, indent=2), encoding="utf-8")
print(met.read_text()[:1400])


In [ ]:
# --- Publish to Drive (latest + immutable archive) -----------------------------
#
to_publish = [
    ("predictions", OUTD / "ocr_outputs.csv"),
    ("metrics", met),
]
for kind, src in to_publish:
    if src is None:
        continue
    p = Path(src)
    if not p.exists():
        print(f"  skip (not produced): {p}")
        continue
    CB.publish("damir", p, kind, paths=paths, run_timestamp=RUN_TS)

print("\nLatest ->", paths.outputs("damir"))
print("Archive ->", paths.run_dir("damir", timestamp=RUN_TS))

In [ ]:
# --- publish the two remaining CSVs + hand off -------------------------------
for f in ["parameter_presence_results.csv", "terms_extraction_results.csv"]:
    CB.publish("damir", OUTD/f, "predictions", paths=paths, run_timestamp=RUN_TS)

up = paths.inputs/"upstream"/"damir"; up.mkdir(parents=True, exist_ok=True)
for f in ["ocr_outputs.csv", "parameter_presence_results.csv", "terms_extraction_results.csv"]:
    shutil.copyfile(OUTD/f, up/f)
print("handed off ->", up)

## Report log — fill this in before you finish

Copy your answers into `presentation/member_reports/damir_report_log.md` in the repo (or paste
them to the integrator). This is the raw material for the group report and slide deck, so be
specific and **honest about what didn't work**.

1. Which OCR engine, and why (EasyOCR vs PaddleOCR vs Tesseract) — did you compare any?
2. Headline CER/WER on the primary set, plus how preprocessing changed them.
3. The secondary invoice eval: your number AND its denominator (~197 of 750 have GT).
4. Where parameter_checker / terms_extraction needed adapting, and any bug you found.
5. Which business parameters are hardest to detect and what that means for Pistac.io readiness.

Also note anything the next stage needs from you, and which figure you'd put on a slide.